# Ting – Aggregazione e statistiche multi-campione

Legge i `ting_batch_summary.csv` di più cartelle, li unisce raggruppando per etichetta assegnata manualmente, e produce:
- **PDF** con boxplot (E0, betaE, tc, R²) e tabella medie ± SD per gruppo
- **CSV** con tutte le statistiche aggregate

> ⚠️ I CSV devono essere stati generati con la versione **aggiornata** di `ting_utils.py`  
> (quella che include le colonne `E0_Pa`, `betaE`, `tc_ms` ecc.).

In [261]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
print('OK')

OK


## ⚙️ CONFIGURAZIONE — modifica solo questa cella

In [262]:
# Scoperta automatica delle cartelle da unire sotto ANALISI/UMANE
# Ogni sottocartella data puo contenere le tipologie rab5 e/o ev
from pathlib import Path

CONCORR_UMANE_ROOT = Path("/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE")
INCLUDE_TYPES = ("rab5", "ev")

# Alias visualizzati in maiuscolo, mantenendo compatibili i filtri legacy.
TYPE_ALIASES = {"rab5": "RAB5", "ev": "EV"}

FOLDERS = []
for date_dir in sorted(CONCORR_UMANE_ROOT.iterdir()):
    if not date_dir.is_dir():
        continue
    for sample_type in INCLUDE_TYPES:
        sample_dir = date_dir / sample_type
        if sample_dir.is_dir():
            FOLDERS.append({
                "path": str(sample_dir),
                "label": f"{date_dir.name}_{TYPE_ALIASES[sample_type]}",
                "sample_type": sample_type,
            })

print(f"Cartelle rilevate automaticamente: {len(FOLDERS)}")
for entry in FOLDERS:
    print(f"  {entry['label']} [{entry['sample_type']}]: {entry['path']}")
# Dove salvare l'output
OUTPUT_DIR = "/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated"
# Filtra solo curve con status == 'ok'
ONLY_OK = True
# Escludi curve con betaE fissato al bound (fit potenzialmente degenere)
EXCLUDE_BETA_AT_BOUND = True
# Escludi curve con tempo tc negativo (fit non fisico / degenerato)
EXCLUDE_NEGATIVE_TC = True
# Escludi curve con R^2 negativo (fit peggiore della media)
EXCLUDE_NEGATIVE_R2 = True
R2_COLUMN_FOR_FILTER = "r2_plr"
# Abilita aggregazione trasversale: tutte le RAB5 insieme e tutte le EV insieme
ENABLE_A1_0R_AGGREGATION = True

# Filtro robusto per curve estremamente fuori distribuzione (MAD-zscore)
ENABLE_EXTREME_CURVE_FILTER = True
EXTREME_FILTER_TARGET = "larger"  # "larger" | "RAB5" | "EV" | "all"
EXTREME_FILTER_COLUMNS = ("E0_Pa", "betaE", "tc_ms", "r2_plr")
EXTREME_FILTER_MAD_Z = 4.5
EXTREME_FILTER_MAX_FRACTION = 0.20
EXTREME_FILTER_MIN_CURVES_PER_MACRO = 60

# Nei boxplot: un punto per ogni curva (non media per cellula)
PLOT_EACH_CURVE_POINT = True
# Nei boxplot: scrivi il nome della cellula vicino a ciascun punto (se disponibile)
LABEL_POINTS_WITH_CELL_NAME = False
# Parametri da includere nel riassunto — chiave: nome colonna CSV, valore: etichetta leggibile
PARAMS = {
    "E0_Pa":          "E₀ [Pa]",
    "betaE":          "βE",
    "tc_ms":          "tc [ms]",
    "r2_plr":         "R² PLR",
}
TABLE_PARAMS = {
    "E0_Pa":          "E₀ [Pa]",
    "betaE":          "βE",
    "tc_ms":          "tc [ms]",
    "r2_plr":         "R² PLR",
}
PLOT_PARAMS = {
    "E0_Pa":          "E₀ [Pa]",
    "betaE":          "βE",
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output in: {OUTPUT_DIR}")

Cartelle rilevate automaticamente: 12
  090626_RAB5 [rab5]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/090626/rab5
  090626_EV [ev]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/090626/ev
  100626_RAB5 [rab5]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/100626/rab5
  100626_EV [ev]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/100626/ev
  130526_EV [ev]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/130526/ev
  140526_EV [ev]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/140526/ev
  150626_RAB5 [rab5]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/150626/rab5
  160626_RAB5 [rab5]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/160626/rab5
  250626_RAB5 [rab5]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/250626/rab5
  250626_EV [ev]: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyN

In [263]:
# Controlli di qualita prima dell'aggregazione
REQUIRE_PLR_MODEL = True
EXCLUDE_BETA_BOUND_CURVES = True
EXCLUDE_TC_BOUND_CURVES = False
EXCLUDE_TOO_SMALL_TC = True
TC_MIN_FOR_AGGREGATION_MS = 1.0
R2_MIN_FOR_AGGREGATION = 0.70
BETA_E_LOWER_BOUND = 0.01
BETA_E_UPPER_BOUND = 0.49
BETA_E_BOUND_TOLERANCE = 0.005

# Il MAD viene disattivato nella cella di caricamento e applicato dopo i controlli.
ENABLE_EXTREME_CURVE_FILTER = False
ENABLE_EXTREME_CURVE_FILTER_AFTER_QUALITY = True

## 1. Scoperta automatica dei CSV

In [264]:
csv_entries = []
for entry in FOLDERS:
    p = os.path.join(entry["path"], "ting_batch_summary.csv")
    if not os.path.isfile(p):
        print(f"[MANCANTE] {p}")
    else:
        csv_entries.append({"path": p, "label": entry["label"]})
        print(f"  OK  [{entry['label']}]  {p}")

if not csv_entries:
    raise FileNotFoundError("Nessun ting_batch_summary.csv trovato. Controlla i percorsi in FOLDERS.")

print(f"\n{len(csv_entries)} cartelle pronte.")

  OK  [090626_RAB5]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/090626/rab5/ting_batch_summary.csv
  OK  [090626_EV]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/090626/ev/ting_batch_summary.csv
  OK  [100626_RAB5]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/100626/rab5/ting_batch_summary.csv
  OK  [100626_EV]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/100626/ev/ting_batch_summary.csv
  OK  [130526_EV]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/130526/ev/ting_batch_summary.csv
  OK  [140526_EV]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/140526/ev/ting_batch_summary.csv
  OK  [150626_RAB5]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/150626/rab5/ting_batch_summary.csv
  OK  [160626_RAB5]  /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/ANALISI/UMANE/160626/rab5/ting_batch_summary.csv
  OK  [250626_RAB5]  /Us

## 2. Caricamento e pulizia

In [265]:
dfs = []

for entry in csv_entries:

    df_tmp = pd.read_csv(entry["path"])
    df_tmp["group"] = entry["label"]
    df_tmp["source_path"] = entry["path"]

    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

print(f"Curve totali caricate: {len(df)}")

# Filtri qualità
if ONLY_OK:
    before = len(df)
    df = df[df["status"].astype(str) == "ok"]
    print(f"Dopo filtro status==ok: {before} → {len(df)}")

if EXCLUDE_BETA_AT_BOUND and "betaE_at_bound" in df.columns:
    before = len(df)
    df = df[df["betaE_at_bound"] != True]
    print(f"Dopo filtro betaE_at_bound: {before} → {len(df)}")

def infer_rab5_ev_group_from_text(text):
    s = str(text).upper().replace("-", "_")
    if "_RAB5" in s or s.endswith("RAB5") or "RAB5_" in s or "/RAB5" in s:
        return "RAB5"
    if "_EV" in s or s.endswith("EV") or "EV_" in s or "/EV" in s:
        return "EV"
    return np.nan

agg_enabled = bool(globals().get("ENABLE_A1_0R_AGGREGATION", False))
if agg_enabled:
    macro_from_label = df["group"].map(infer_rab5_ev_group_from_text)
    macro_from_path = df["source_path"].map(infer_rab5_ev_group_from_text) if "source_path" in df.columns else np.nan
    df["macro_group"] = macro_from_label.fillna(macro_from_path)

    macro_counts = df["macro_group"].value_counts(dropna=True)
    print("\nCurve aggregate RAB5/EV:")
    if len(macro_counts) == 0:
        print("  Nessuna curva riconosciuta come RAB5 o EV dalle label/path.")
    else:
        for name, count in macro_counts.items():
            print(f"  {name}: {count}")

# Converti in numerico e controlla colonne disponibili
available = {}
for col, label in PARAMS.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        available[col] = label
    elif col == "young_hertz_pa" and "young_hertz_kpa" in df.columns:
        df["young_hertz_kpa"] = pd.to_numeric(df["young_hertz_kpa"], errors="coerce")
        df[col] = df["young_hertz_kpa"] * 1000.0
        available[col] = label
    else:
        print(f"  [mancante] {col} — riesegui il batch con ting_utils.py aggiornato")

if EXCLUDE_NEGATIVE_TC and "tc_ms" in df.columns:
    tc_vals = pd.to_numeric(df["tc_ms"], errors="coerce")
    neg_tc = tc_vals < 0
    if bool(neg_tc.any()):
        n_neg = int(neg_tc.sum())
        print(f"  [filtro] tc_ms negativi rimossi: {n_neg}")
        df = df.loc[~neg_tc].copy()

if EXCLUDE_NEGATIVE_R2 and R2_COLUMN_FOR_FILTER in df.columns:
    r2_vals = pd.to_numeric(df[R2_COLUMN_FOR_FILTER], errors="coerce")
    neg_r2 = r2_vals < 0
    if bool(neg_r2.any()):
        n_neg_r2 = int(neg_r2.sum())
        print(f"  [filtro] {R2_COLUMN_FOR_FILTER} negativi rimossi: {n_neg_r2}")
        df = df.loc[~neg_r2].copy()

# Filtro robusto curve estreme via MAD-zscore
if agg_enabled and bool(globals().get("ENABLE_EXTREME_CURVE_FILTER", False)) and "macro_group" in df.columns:
    cols_filter = [c for c in globals().get("EXTREME_FILTER_COLUMNS", ()) if c in df.columns]
    target_mode = str(globals().get("EXTREME_FILTER_TARGET", "larger")).upper()
    z_thr = float(globals().get("EXTREME_FILTER_MAD_Z", 4.5))
    max_fraction = float(globals().get("EXTREME_FILTER_MAX_FRACTION", 0.20))
    min_curves = int(globals().get("EXTREME_FILTER_MIN_CURVES_PER_MACRO", 60))

    macro_counts_now = df["macro_group"].value_counts(dropna=True)
    target_macros = []
    if len(macro_counts_now) > 0 and cols_filter:
        if target_mode == "LARGER":
            target_macros = [str(macro_counts_now.idxmax())]
        elif target_mode in ("RAB5", "EV"):
            target_macros = [target_mode]
        elif target_mode == "ALL":
            target_macros = [str(x) for x in macro_counts_now.index.tolist()]

    def _robust_z(vals):
        vals = np.asarray(vals, dtype=float)
        med = np.nanmedian(vals)
        mad = np.nanmedian(np.abs(vals - med))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-12:
            return np.zeros_like(vals, dtype=float)
        return (vals - med) / scale

    total_removed = 0
    for macro_name in target_macros:
        idx = df.index[df["macro_group"].astype(str) == macro_name]
        if len(idx) <= min_curves:
            continue

        sub = df.loc[idx, cols_filter].apply(pd.to_numeric, errors="coerce")
        z_cols = []
        for c in cols_filter:
            z = _robust_z(sub[c].values)
            z_cols.append(np.abs(z))
        if not z_cols:
            continue

        z_matrix = np.vstack(z_cols).T
        score = np.nanmax(z_matrix, axis=1)
        score = np.where(np.isfinite(score), score, -np.inf)
        cand_local = np.where(score > z_thr)[0]
        if cand_local.size == 0:
            continue

        max_remove_by_fraction = int(np.floor(max_fraction * len(idx)))
        max_remove_by_min = max(0, len(idx) - min_curves)
        max_remove = min(cand_local.size, max_remove_by_fraction, max_remove_by_min)
        if max_remove <= 0:
            continue

        order = np.argsort(score[cand_local])[::-1]
        to_remove_local = cand_local[order[:max_remove]]
        remove_idx = idx[to_remove_local]
        df = df.drop(index=remove_idx)
        total_removed += len(remove_idx)
        print(f"  [filtro estremi] {macro_name}: rimosse {len(remove_idx)} curve (thr>|z|>{z_thr}, colonne={cols_filter})")

    if total_removed == 0:
        print("  [filtro estremi] nessuna curva rimossa")
    else:
        print(f"  [filtro estremi] totale curve rimosse: {total_removed}")

print(f"\nParametri disponibili: {list(available.keys())}")
print(f"Gruppi trovati: {sorted(df['group'].unique())}")
if "macro_group" in df.columns:
    print("Macro-gruppi dopo filtri:", df["macro_group"].value_counts(dropna=True).to_dict())
df.head(3)

Curve totali caricate: 486
Dopo filtro status==ok: 486 → 482

Curve aggregate RAB5/EV:
  RAB5: 256
  EV: 226
  [filtro] r2_plr negativi rimossi: 22

Parametri disponibili: ['E0_Pa', 'betaE', 'tc_ms', 'r2_plr']
Gruppi trovati: ['090626_EV', '090626_RAB5', '100626_EV', '100626_RAB5', '130526_EV', '140526_EV', '150626_RAB5', '160626_RAB5', '250626_EV', '250626_RAB5', '260626_EV', '260626_RAB5']
Macro-gruppi dopo filtri: {'RAB5': 252, 'EV': 208}


,curve,cell_folder,fd_folder,path,status,selected_model,selection_reason,E0_Pa,betaE,tc_ms,...,tau2_ms,rmse_gm2_pN,r2_gm2,retrace_trim_points,closure_refined,contact_strategy,force_drag_applied,group,source_path,macro_group
0,Area1_FD-0000,cell01,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,ok,SLS,BIC=-113639.46 | R2=0.8909 | scelta penalizzan...,253.565200,0.010001,1.562099e-03,...,NaN,NaN,NaN,0.0,False,current_default,True,090626_RAB5,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,RAB5
1,Area1_FD-0001,cell01,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,ok,SLS,BIC=-110616.34 | R2=0.7840 | scelta penalizzan...,131.929433,0.010000,1.316334e-07,...,NaN,NaN,NaN,0.0,True,current_default,True,090626_RAB5,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,RAB5
2,Area1_FD-0002,cell01,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,ok,SLS,BIC=-180796.18 | R2=0.9324 | scelta penalizzan...,149.486189,0.025117,6.233456e-01,...,NaN,NaN,NaN,0.0,False,current_default,True,090626_RAB5,/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI...,RAB5


## 2bis. Controllo bound del fit e filtro R²

I bound di βE vengono conteggiati per condizione prima dell'esclusione. Il filtro R² è applicato prima delle statistiche, dei test Mann–Whitney e dei grafici.

In [266]:
quality_df = df.copy()

if REQUIRE_PLR_MODEL:
    model_keep = quality_df["selected_model"].astype(str).str.upper().eq("PLR")
    model_counts = quality_df.groupby("macro_group", dropna=False)["selected_model"].agg(
        curve_totali="size",
        PLR_selezionato=lambda values: values.astype(str).str.upper().eq("PLR").sum(),
    ).reset_index()
    print("Modello selezionato per condizione:")
    display(model_counts)
    quality_df = quality_df.loc[model_keep].copy()

if "betaE_at_bound" in quality_df.columns:
    quality_df["betaE_at_bound_check"] = quality_df["betaE_at_bound"].fillna(False).astype(bool)
elif "betaE" in quality_df.columns:
    beta_values = pd.to_numeric(quality_df["betaE"], errors="coerce")
    quality_df["betaE_at_bound_check"] = (
        beta_values.notna()
        & (
            (np.abs(beta_values - BETA_E_LOWER_BOUND) <= BETA_E_BOUND_TOLERANCE)
            | (np.abs(beta_values - BETA_E_UPPER_BOUND) <= BETA_E_BOUND_TOLERANCE)
        )
    )
else:
    quality_df["betaE_at_bound_check"] = False

bound_log = (
    quality_df.groupby("macro_group", dropna=False)["betaE_at_bound_check"]
    .agg(curve_totali="size", betaE_al_bound="sum")
    .reset_index()
)
bound_log["percentuale_bound"] = 100.0 * bound_log["betaE_al_bound"] / bound_log["curve_totali"].clip(lower=1)
print("Controllo curve al bound βE dopo il filtro PLR:")
display(bound_log)

before_quality_filter = len(quality_df)
if EXCLUDE_BETA_BOUND_CURVES:
    quality_df = quality_df.loc[~quality_df["betaE_at_bound_check"]].copy()

if EXCLUDE_TOO_SMALL_TC:
    tc_values = pd.to_numeric(quality_df["tc_ms"], errors="coerce")
    tc_quality_log = (
        quality_df.assign(tc_too_small=tc_values < float(TC_MIN_FOR_AGGREGATION_MS))
        .groupby("macro_group", dropna=False)["tc_too_small"]
        .agg(curve_totali="size", tc_sotto_soglia="sum")
        .reset_index()
    )
    tc_quality_log["percentuale_sotto_soglia"] = 100.0 * tc_quality_log["tc_sotto_soglia"] / tc_quality_log["curve_totali"].clip(lower=1)
    print(f"Controllo tc: esclusione delle curve con tc < {TC_MIN_FOR_AGGREGATION_MS} ms")
    display(tc_quality_log)
    quality_df = quality_df.loc[tc_values >= float(TC_MIN_FOR_AGGREGATION_MS)].copy()

r2_values = pd.to_numeric(quality_df[R2_COLUMN_FOR_FILTER], errors="coerce")
r2_keep = r2_values.notna() & (r2_values >= float(R2_MIN_FOR_AGGREGATION))
quality_df = quality_df.loc[r2_keep].copy()

after_quality_filter = len(quality_df)
print(
    f"Filtro qualita prima dell'aggregazione: {before_quality_filter} → {after_quality_filter} curve "
    f"(modello=PLR, βE/tc esclusi, R² >= {R2_MIN_FOR_AGGREGATION})"
)
print("Curve mantenute per condizione:")
display(quality_df["macro_group"].value_counts(dropna=False).rename("curve_mantenute").to_frame())

df = quality_df.drop(columns=["betaE_at_bound_check"], errors="ignore")

Modello selezionato per condizione:


,macro_group,curve_totali,PLR_selezionato
0,EV,208,172
1,RAB5,252,216


Controllo curve al bound βE dopo il filtro PLR:


,macro_group,curve_totali,betaE_al_bound,percentuale_bound
0,EV,172,11,6.395349
1,RAB5,216,16,7.407407


Controllo tc: esclusione delle curve con tc < 1.0 ms


,macro_group,curve_totali,tc_sotto_soglia,percentuale_sotto_soglia
0,EV,161,9,5.590062
1,RAB5,200,6,3.000000


Filtro qualita prima dell'aggregazione: 388 → 306 curve (modello=PLR, βE/tc esclusi, R² >= 0.7)
Curve mantenute per condizione:


,curve_mantenute
macro_group,
RAB5,177
EV,129


## 2ter. Filtro MAD sugli outlier dopo i controlli di qualità

Il filtro MAD viene applicato ora, dopo il filtro obbligatorio PLR, l'esclusione dei bound e il filtro R². Non viene applicata alcuna soglia fissa su E₀.

In [267]:
if ENABLE_EXTREME_CURVE_FILTER_AFTER_QUALITY and "macro_group" in df.columns:
    cols_filter = [c for c in EXTREME_FILTER_COLUMNS if c in df.columns and c != "E0_Pa"]
    cols_filter = ["E0_Pa"] + cols_filter if "E0_Pa" in df.columns else cols_filter
    target_mode = str(EXTREME_FILTER_TARGET).upper()
    target_macros = []
    macro_counts_now = df["macro_group"].value_counts(dropna=True)
    if target_mode == "LARGER" and len(macro_counts_now) > 0:
        target_macros = [str(macro_counts_now.idxmax())]
    elif target_mode in ("RAB5", "EV"):
        target_macros = [target_mode]
    elif target_mode == "ALL":
        target_macros = [str(x) for x in macro_counts_now.index]

    def _robust_z_after_quality(values):
        values = np.asarray(values, dtype=float)
        median = np.nanmedian(values)
        mad = np.nanmedian(np.abs(values - median))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-12:
            return np.zeros_like(values, dtype=float)
        return (values - median) / scale

    mad_removed = 0
    for macro_name in target_macros:
        indexes = df.index[df["macro_group"].astype(str) == macro_name]
        if len(indexes) <= EXTREME_FILTER_MIN_CURVES_PER_MACRO:
            continue
        values = df.loc[indexes, cols_filter].apply(pd.to_numeric, errors="coerce")
        scores = np.nanmax(np.vstack([np.abs(_robust_z_after_quality(values[col].values)) for col in cols_filter]).T, axis=1)
        candidates = np.where(np.isfinite(scores) & (scores > EXTREME_FILTER_MAD_Z))[0]
        max_remove = min(
            len(candidates),
            int(np.floor(EXTREME_FILTER_MAX_FRACTION * len(indexes))),
            max(0, len(indexes) - EXTREME_FILTER_MIN_CURVES_PER_MACRO),
        )
        if max_remove <= 0:
            continue
        remove_indexes = indexes[candidates[np.argsort(scores[candidates])[::-1][:max_remove]]]
        df = df.drop(index=remove_indexes)
        mad_removed += len(remove_indexes)
        print(f"[MAD dopo qualità] {macro_name}: rimosse {len(remove_indexes)} curve")
    print(f"Totale rimosse dal MAD dopo qualità: {mad_removed}")

[MAD dopo qualità] RAB5: rimosse 10 curve
Totale rimosse dal MAD dopo qualità: 10


## 2quater. Audit R² e sensibilità al giorno 100626

L'audit verifica che il dataframe usato per l'aggregazione rispetti davvero la soglia R². La sensibilità esclude entrambe le condizioni del giorno 100626 senza modificare il dataset principale.

In [268]:
audit_r2 = (
    df.groupby("group", sort=True)[R2_COLUMN_FOR_FILTER]
    .agg(N_curve="count", R2_min="min", R2_media="mean", R2_max="max")
    .reset_index()
)
audit_r2["soglia_rispettata"] = audit_r2["R2_min"] >= float(R2_MIN_FOR_AGGREGATION)
print(f"Audit R² dopo tutti i filtri: soglia R² >= {R2_MIN_FOR_AGGREGATION}")
display(audit_r2)
if not bool(audit_r2["soglia_rispettata"].all()):
    raise AssertionError("Il dataframe finale contiene almeno un gruppo con R² sotto soglia.")


def cell_level_values(frame, parameter):
    cell_key = frame["group"].astype(str)
    if "cell_folder" in frame.columns:
        cell_key = cell_key + "::" + frame["cell_folder"].astype(str)
    tmp = frame.copy()
    tmp["cell_key"] = cell_key
    return (
        tmp.groupby(["macro_group", "cell_key"], dropna=True)[parameter]
        .mean(numeric_only=True)
        .reset_index()
    )


def mann_whitney_cell_sensitivity(frame, parameter):
    values = cell_level_values(frame, parameter)
    rab5 = values.loc[values["macro_group"] == "RAB5", parameter].dropna().to_numpy(dtype=float)
    ev = values.loc[values["macro_group"] == "EV", parameter].dropna().to_numpy(dtype=float)
    if len(rab5) == 0 or len(ev) == 0:
        return {"parameter": parameter, "n_RAB5": len(rab5), "n_EV": len(ev), "U": np.nan, "p_value": np.nan, "r_biseriale": np.nan}
    u_stat, p_value = mannwhitneyu(rab5, ev, alternative="two-sided", method="auto")
    return {
        "parameter": parameter,
        "n_RAB5": len(rab5),
        "n_EV": len(ev),
        "U": float(u_stat),
        "p_value": float(p_value),
        "r_biseriale": float(2.0 * u_stat / (len(rab5) * len(ev)) - 1.0),
    }

sensitivity_rows = []
for label, frame in [("tutti i giorni", df), ("escluso 100626", df[~df["group"].astype(str).str.startswith("100626_")].copy())]:
    for parameter in ["E0_Pa", "betaE", "tc_ms", "r2_plr"]:
        result = mann_whitney_cell_sensitivity(frame, parameter)
        result["dataset"] = label
        sensitivity_rows.append(result)

sensitivity_results = pd.DataFrame(sensitivity_rows)[
    ["dataset", "parameter", "n_RAB5", "n_EV", "U", "p_value", "r_biseriale"]
]
print("Sensibilità Mann–Whitney U a livello cellula, con e senza 100626:")
display(sensitivity_results)

Audit R² dopo tutti i filtri: soglia R² >= 0.7


,group,N_curve,R2_min,R2_media,R2_max,soglia_rispettata
0,090626_EV,40,0.816008,0.939743,0.982331,True
1,090626_RAB5,3,0.894018,0.914096,0.929277,True
2,100626_EV,4,0.701087,0.789394,0.893370,True
3,100626_RAB5,36,0.738286,0.849155,0.933301,True
4,130526_EV,19,0.833080,0.916337,0.981971,True
5,140526_EV,5,0.772735,0.877027,0.949119,True
6,150626_RAB5,35,0.796730,0.958606,0.987667,True
7,160626_RAB5,30,0.774157,0.948359,0.995242,True
8,250626_EV,26,0.823967,0.922902,0.986026,True
9,250626_RAB5,38,0.765220,0.906745,0.985668,True


Sensibilità Mann–Whitney U a livello cellula, con e senza 100626:


,dataset,parameter,n_RAB5,n_EV,U,p_value,r_biseriale
0,tutti i giorni,E0_Pa,25,22,374.0,0.035726,0.360000
1,tutti i giorni,betaE,25,22,289.0,0.773484,0.050909
2,tutti i giorni,tc_ms,25,22,198.0,0.102893,-0.280000
3,tutti i giorni,r2_plr,25,22,268.0,0.889782,-0.025455
4,escluso 100626,E0_Pa,16,20,241.0,0.010384,0.506250
5,escluso 100626,betaE,16,20,124.0,0.258405,-0.225000
6,escluso 100626,tc_ms,16,20,98.0,0.050242,-0.387500
7,escluso 100626,r2_plr,16,20,203.0,0.176050,0.268750


## 2quinquies. Audit numerosità per cellula e giornate anomale

Questo controllo non elimina automaticamente le giornate: documenta la numerosità interna e rende visibili i gruppi da verificare sulle curve grezze.

In [269]:
cell_key_audit = df["group"].astype(str)
if "cell_folder" in df.columns:
    cell_key_audit = cell_key_audit + "::" + df["cell_folder"].astype(str)

cell_audit = (
    df.assign(cell_key=cell_key_audit)
    .groupby(["macro_group", "group", "cell_folder", "cell_key"], dropna=True)
    .agg(
        N_curve=("curve", "size"),
        E0_Pa_media=("E0_Pa", "mean"),
        betaE_media=("betaE", "mean"),
        tc_ms_media=("tc_ms", "mean"),
        r2_plr_media=("r2_plr", "mean"),
        r2_plr_min=("r2_plr", "min"),
    )
    .reset_index()
)

print("Numerosità per singola cellula:")
display(
    cell_audit.groupby("macro_group")["cell_key"]
    .nunique()
    .rename("N cellule")
    .to_frame()
)
print("Range N curve per cellula:")
display(
    cell_audit.groupby("macro_group")["N_curve"]
    .agg(N_cellule="count", N_min="min", N_mediana="median", N_max="max")
    .reset_index()
)
print("Giornate con poche cellule o qualità relativamente più bassa:")
display(
    audit_r2.sort_values(["R2_media", "N_curve"])
    .head(6)
)
print("Cellule con E0 più alto/basso, da verificare sulle curve grezze:")
display(
    cell_audit.sort_values("E0_Pa_media")
    [["macro_group", "group", "cell_folder", "N_curve", "E0_Pa_media", "betaE_media", "tc_ms_media", "r2_plr_media"]]
    .head(10)
)


Numerosità per singola cellula:


,N cellule
macro_group,
EV,22
RAB5,25


Range N curve per cellula:


,macro_group,N_cellule,N_min,N_mediana,N_max
0,EV,22,1,4.0,26
1,RAB5,25,1,5.0,38


Giornate con poche cellule o qualità relativamente più bassa:


,group,N_curve,R2_min,R2_media,R2_max,soglia_rispettata
2,100626_EV,4,0.701087,0.789394,0.893370,True
3,100626_RAB5,36,0.738286,0.849155,0.933301,True
5,140526_EV,5,0.772735,0.877027,0.949119,True
11,260626_RAB5,25,0.762804,0.894353,0.978337,True
10,260626_EV,35,0.709808,0.901659,0.991226,True
9,250626_RAB5,38,0.765220,0.906745,0.985668,True


Cellule con E0 più alto/basso, da verificare sulle curve grezze:


,macro_group,group,cell_folder,N_curve,E0_Pa_media,betaE_media,tc_ms_media,r2_plr_media
3,EV,090626_EV,cell04 10.45.19,3,15.925639,0.268556,59.634482,0.867555
11,EV,100626_EV,cell09,2,26.099201,0.369217,19.000000,0.781559
1,EV,090626_EV,cell02,1,30.552842,0.327501,32.000000,0.860019
10,EV,100626_EV,cell05,2,34.985868,0.326443,38.762852,0.797229
44,RAB5,260626_RAB5,cell01,6,53.554709,0.356292,20.505324,0.886205
23,RAB5,090626_RAB5,cell06,1,56.234217,0.103208,38.000000,0.918992
6,EV,090626_EV,cell07,4,58.643769,0.215763,30.999989,0.935702
29,RAB5,100626_RAB5,cell07,7,59.989939,0.296972,27.285709,0.859369
25,RAB5,100626_RAB5,cell03,4,60.809390,0.301624,27.598509,0.797138
4,EV,090626_EV,cell05,5,62.562263,0.173028,36.199981,0.954934


## 3. Statistiche per giornata + controllo per cellula

La tabella per giornata è descrittiva. Le inferenze Mann–Whitney usano invece la media per singola cellula (`giornata + cell_folder`), per evitare pseudoreplicazione.

In [270]:
rows_stats = []
for group, gdf in df.groupby("group", sort=True):
    row = {"Giornata": group, "N curve": len(gdf)}
    for col, label in TABLE_PARAMS.items():
        vals = gdf[col].dropna()
        if len(vals) == 0:
            row[f"{label} media"] = None
            row[f"{label} SD"] = None
        else:
            row[f"{label} media"] = round(float(np.mean(vals)), 4)
            row[f"{label} SD"] = round(float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0, 4)
    rows_stats.append(row)

# Righe finali: medie globali descrittive separate per tipo (RAB5 e EV)
agg_enabled = bool(globals().get("ENABLE_A1_0R_AGGREGATION", False))
if agg_enabled and "macro_group" in df.columns:
    for macro_name in ["RAB5", "EV"]:
        sub = df[df["macro_group"] == macro_name]
        if len(sub) == 0:
            continue
        row_global = {"Giornata": f"─── MEDIA GLOBALE {macro_name} ───", "N curve": len(sub)}
        for col, label in TABLE_PARAMS.items():
            vals_all = sub[col].dropna()
            if len(vals_all) == 0:
                row_global[f"{label} media"] = None
                row_global[f"{label} SD"] = None
            else:
                row_global[f"{label} media"] = round(float(np.mean(vals_all)), 4)
                row_global[f"{label} SD"] = round(float(np.std(vals_all, ddof=1)) if len(vals_all) > 1 else 0.0, 4)
        rows_stats.append(row_global)

df_stats = pd.DataFrame(rows_stats)
print("Statistiche descrittive per giornata:")
display(df_stats)

# Controllo descrittivo coerente con l'unità indipendente del test.
cell_key = df["group"].astype(str)
if "cell_folder" in df.columns:
    cell_key = cell_key + "::" + df["cell_folder"].astype(str)
cell_stats_source = df.copy()
cell_stats_source["cell_key"] = cell_key
cell_stats = (
    cell_stats_source.groupby(["macro_group", "cell_key", "group", "cell_folder"], dropna=True)
    .agg(
        N_curve=("curve", "size"),
        E0_Pa_media=("E0_Pa", "mean"),
        betaE_media=("betaE", "mean"),
        tc_ms_media=("tc_ms", "mean"),
        r2_plr_media=("r2_plr", "mean"),
    )
    .reset_index()
)
print("Statistiche descrittive per singola cellula, usate anche nel test principale:")
display(cell_stats.sort_values(["macro_group", "group", "cell_folder"]))

# Alias per mantenere compatibili le pagine PDF già costruite sul nome storico.
df_stats = df_stats.rename(columns={"Giornata": "Cellula"})

Statistiche descrittive per giornata:


,Giornata,N curve,E₀ [Pa] media,E₀ [Pa] SD,βE media,βE SD,tc [ms] media,tc [ms] SD,R² PLR media,R² PLR SD
0,090626_EV,40,78.1220,51.8961,0.1830,0.0669,32.0915,10.8986,0.9397,0.0414
1,090626_RAB5,3,61.0582,18.4973,0.1522,0.0627,29.2481,9.7467,0.9141,0.0181
2,100626_EV,4,30.5425,24.9664,0.3478,0.0592,28.8814,17.8191,0.7894,0.0846
3,100626_RAB5,36,89.1177,64.4862,0.2811,0.0673,24.3161,6.8447,0.8492,0.0506
4,130526_EV,19,94.1875,32.3748,0.1794,0.0520,22.4488,3.0287,0.9163,0.0402
5,140526_EV,5,79.4782,8.5328,0.1999,0.0708,21.8000,1.3038,0.8770,0.0675
6,150626_RAB5,35,153.3526,61.7853,0.1463,0.0486,22.7328,4.4500,0.9586,0.0353
7,160626_RAB5,30,177.0996,90.5345,0.1743,0.0428,20.7522,2.0715,0.9484,0.0454
8,250626_EV,26,110.7408,67.6163,0.1778,0.0478,23.2578,3.8001,0.9229,0.0427
9,250626_RAB5,38,98.0344,37.6121,0.2044,0.0502,21.0584,2.3858,0.9067,0.0575


Statistiche descrittive per singola cellula, usate anche nel test principale:


,macro_group,cell_key,group,cell_folder,N_curve,E0_Pa_media,betaE_media,tc_ms_media,r2_plr_media
0,EV,090626_EV::cell01,090626_EV,cell01,1,213.407388,0.061926,3.851307,0.932697
1,EV,090626_EV::cell02,090626_EV,cell02,1,30.552842,0.327501,32.000000,0.860019
2,EV,090626_EV::cell03,090626_EV,cell03,2,96.044515,0.166663,27.999931,0.858627
3,EV,090626_EV::cell04 10.45.19,090626_EV,cell04 10.45.19,3,15.925639,0.268556,59.634482,0.867555
4,EV,090626_EV::cell05,090626_EV,cell05,5,62.562263,0.173028,36.199981,0.954934
5,EV,090626_EV::cell06,090626_EV,cell06,8,84.833765,0.171097,29.594903,0.950407
6,EV,090626_EV::cell07,090626_EV,cell07,4,58.643769,0.215763,30.999989,0.935702
7,EV,090626_EV::cell08,090626_EV,cell08,5,72.037830,0.182660,30.200000,0.959634
8,EV,090626_EV::cell09,090626_EV,cell09,4,115.207615,0.167521,26.500000,0.935137
9,EV,090626_EV::cell10,090626_EV,cell10,7,84.853783,0.158686,30.592260,0.973946


## 3bis. Confronto RAB5 vs EV: Mann–Whitney U

Il test principale è calcolato sulle medie per cellula, per evitare pseudoreplicazione. Viene riportato anche il confronto esplorativo sulle curve singole.

In [271]:
from scipy.stats import mannwhitneyu

MW_PARAMETERS = {
    "E0_Pa": "E₀ [Pa]",
    "betaE": "βE",
    "tc_ms": "tc [ms]",
    "r2_plr": "R² PLR",
}


def rank_biserial_effect(x, y, u_stat):
    n_x = len(x)
    n_y = len(y)
    return float(2.0 * u_stat / (n_x * n_y) - 1.0)


def mann_whitney_row(x, y, parameter, level):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna().to_numpy(dtype=float)
    if len(x) == 0 or len(y) == 0:
        return {
            "parameter": parameter,
            "level": level,
            "n_RAB5": len(x),
            "n_EV": len(y),
            "median_RAB5": np.nan,
            "median_EV": np.nan,
            "U": np.nan,
            "p_value": np.nan,
            "rank_biserial_r": np.nan,
        }
    u_stat, p_value = mannwhitneyu(x, y, alternative="two-sided", method="auto")
    return {
        "parameter": parameter,
        "level": level,
        "n_RAB5": len(x),
        "n_EV": len(y),
        "median_RAB5": float(np.median(x)),
        "median_EV": float(np.median(y)),
        "U": float(u_stat),
        "p_value": float(p_value),
        "rank_biserial_r": rank_biserial_effect(x, y, u_stat),
    }


mw_rows = []
if "macro_group" in df.columns:
    # Analisi esplorativa: ogni curva viene trattata come osservazione.
    for parameter, label in MW_PARAMETERS.items():
        if parameter not in df.columns:
            continue
        rab5_values = df.loc[df["macro_group"] == "RAB5", parameter]
        ev_values = df.loc[df["macro_group"] == "EV", parameter]
        row = mann_whitney_row(rab5_values, ev_values, label, "curve")
        row["parameter_code"] = parameter
        mw_rows.append(row)

    # Analisi principale: una media per cellula e condizione.
    cell_key = df["group"].astype(str)
    if "cell_folder" in df.columns:
        cell_key = cell_key + "::" + df["cell_folder"].astype(str)
    cell_df = df.copy()
    cell_df["cell_key"] = cell_key
    cell_means = (
        cell_df.groupby(["macro_group", "cell_key"], dropna=True)[list(MW_PARAMETERS)]
        .mean(numeric_only=True)
        .reset_index()
    )

    for parameter, label in MW_PARAMETERS.items():
        if parameter not in cell_means.columns:
            continue
        rab5_values = cell_means.loc[cell_means["macro_group"] == "RAB5", parameter]
        ev_values = cell_means.loc[cell_means["macro_group"] == "EV", parameter]
        row = mann_whitney_row(rab5_values, ev_values, label, "cellula")
        row["parameter_code"] = parameter
        mw_rows.append(row)

mw_results = pd.DataFrame(mw_rows)
if not mw_results.empty:
    mw_results = mw_results[
        ["parameter_code", "parameter", "level", "n_RAB5", "n_EV", "median_RAB5", "median_EV", "U", "p_value", "rank_biserial_r"]
    ]
    print("Mann–Whitney U: RAB5 vs EV")
    display(mw_results)
    print("Il livello 'cellula' è quello da usare per le conclusioni inferenziali; il livello 'curve' è esplorativo.")
else:
    print("Mann–Whitney U non eseguito: macro_group o parametri mancanti.")

Mann–Whitney U: RAB5 vs EV


,parameter_code,parameter,level,n_RAB5,n_EV,median_RAB5,median_EV,U,p_value,rank_biserial_r
0,E0_Pa,E₀ [Pa],curve,167,129,106.896797,82.959736,12970.0,0.002611,0.204103
1,betaE,βE,curve,167,129,0.193015,0.170880,12287.0,0.038007,0.140695
2,tc_ms,tc [ms],curve,167,129,21.999999,23.000000,7664.0,0.000021,-0.288493
3,r2_plr,R² PLR,curve,167,129,0.928202,0.929743,10542.0,0.753814,-0.021306
4,E0_Pa,E₀ [Pa],cellula,25,22,139.341043,84.843774,374.0,0.035726,0.360000
5,betaE,βE,cellula,25,22,0.187692,0.180236,289.0,0.773484,0.050909
6,tc_ms,tc [ms],cellula,25,22,22.363303,24.886197,198.0,0.102893,-0.280000
7,r2_plr,R² PLR,cellula,25,22,0.903392,0.914518,268.0,0.889782,-0.025455


Il livello 'cellula' è quello da usare per le conclusioni inferenziali; il livello 'curve' è esplorativo.


## 4. Salvataggio CSV

In [272]:
path_day_stats = os.path.join(OUTPUT_DIR, "ting_summary_per_giornata_RAB5_EV.csv")
df_stats.to_csv(path_day_stats, index=False)
print(f"Statistiche per giornata: {path_day_stats}")

path_cell_stats = os.path.join(OUTPUT_DIR, "ting_summary_per_cellula_RAB5_EV.csv")
cell_stats.to_csv(path_cell_stats, index=False)
print(f"Statistiche per cellula: {path_cell_stats}")

path_all = os.path.join(OUTPUT_DIR, "ting_summary_tutte_le_curve_RAB5_EV.csv")
df.to_csv(path_all, index=False)
print(f"Tutte le curve filtrate: {path_all}")

path_audit = os.path.join(OUTPUT_DIR, "ting_quality_audit_RAB5_EV.csv")
audit_r2.to_csv(path_audit, index=False)
print(f"Audit qualità R²: {path_audit}")

Statistiche per giornata: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_per_giornata_RAB5_EV.csv
Statistiche per cellula: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_per_cellula_RAB5_EV.csv
Tutte le curve filtrate: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_tutte_le_curve_RAB5_EV.csv
Audit qualità R²: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_quality_audit_RAB5_EV.csv


## 5. PDF — tabella medie ± SD + boxplot per parametro

In [273]:
groups = sorted(df["group"].unique())
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colors = {g: palette[i % len(palette)] for i, g in enumerate(groups)}
pdf_path = os.path.join(OUTPUT_DIR, "ting_summary_report_RAB5_EV.pdf")
np.random.seed(0)


def add_table_page(pdf, table_df, title):
    headers = list(table_df.columns)
    cell_text = [
        [
            str(v) if v is not None and not (isinstance(v, float) and np.isnan(v)) else "—"
            for v in row
        ]
        for row in table_df.itertuples(index=False)
    ]
    fig, ax = plt.subplots(figsize=(max(10, len(headers) * 1.8), max(3, len(table_df) * 0.5 + 1.5)))
    ax.axis("off")
    ax.set_title(title, fontsize=13, pad=14)
    table = ax.table(cellText=cell_text, colLabels=headers, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.0, 1.5)
    table.auto_set_column_width(list(range(len(headers))))
    for (row_index, _), cell in table.get_celld().items():
        if row_index == 0:
            cell.set_facecolor("#DDEEFF")
            cell.set_text_props(weight="bold")
        elif row_index > 0 and cell_text[row_index - 1][0].startswith("─── MEDIA GLOBALE"):
            cell.set_facecolor("#FFF3CD")
            cell.set_text_props(weight="bold")
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_boxplot_page(pdf, df_sub, groups_sub, col, label, title):
    data = [df_sub[df_sub["group"] == g][col].dropna().values for g in groups_sub]
    fig, ax = plt.subplots(figsize=(max(6, len(groups_sub) * 1.5), 5))
    boxplot = ax.boxplot(data, patch_artist=True, widths=0.5, medianprops=dict(color="black", linewidth=2))
    for patch, group_name in zip(boxplot["boxes"], groups_sub):
        patch.set_facecolor(colors[group_name])
        patch.set_alpha(0.6)
    for index, group_name in enumerate(groups_sub, start=1):
        values = df_sub[df_sub["group"] == group_name][col].dropna().values
        jitter = np.random.uniform(-0.15, 0.15, size=len(values))
        ax.scatter(np.full(len(values), index) + jitter, values, color=colors[group_name], alpha=0.8, s=40, zorder=3, edgecolors="white", linewidths=0.5)
    ax.set_xticks(range(1, len(groups_sub) + 1))
    ax.set_xticklabels(groups_sub, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


with PdfPages(pdf_path) as pdf:
    add_table_page(pdf, df_stats, f"Riassunto Ting per giornata — {' | '.join(e['label'] for e in csv_entries)}")
    for col, label in PLOT_PARAMS.items():
        add_boxplot_page(pdf, df, groups, col, label, label)

    if "macro_group" in df.columns:
        for macro_name in ["RAB5", "EV"]:
            sub = df[df["macro_group"] == macro_name].copy()
            if len(sub) == 0:
                continue
            macro_stats = df_stats[df_stats["Cellula"].astype(str).str.contains(macro_name, na=False)]
            add_table_page(pdf, macro_stats, f"Riassunto Ting {macro_name} per giornata")
            groups_sub = sorted(sub["group"].unique())
            for col, label in PLOT_PARAMS.items():
                if groups_sub:
                    add_boxplot_page(pdf, sub, groups_sub, col, label, f"{label} — {macro_name}")

    # Audit qualità e unità indipendente usata dal test.
    audit_pdf = audit_r2.copy()
    audit_pdf["R2_min"] = audit_pdf["R2_min"].round(4)
    audit_pdf["R2_media"] = audit_pdf["R2_media"].round(4)
    audit_pdf["R2_max"] = audit_pdf["R2_max"].round(4)
    add_table_page(pdf, audit_pdf, f"Audit qualità — R² PLR >= {R2_MIN_FOR_AGGREGATION}")

    cell_pdf = cell_audit[[
        "macro_group", "group", "cell_folder", "N_curve",
        "E0_Pa_media", "betaE_media", "tc_ms_media", "r2_plr_media",
    ]].copy()
    cell_pdf = cell_pdf.rename(columns={
        "macro_group": "Condizione",
        "group": "Giornata",
        "cell_folder": "Cellula",
        "N_curve": "N curve",
        "E0_Pa_media": "E0 media [Pa]",
        "betaE_media": "betaE media",
        "tc_ms_media": "tc media [ms]",
        "r2_plr_media": "R2 PLR media",
    }).round(4)
    add_table_page(pdf, cell_pdf, "Statistiche per singola cellula — unità del test")

    if "sensitivity_results" in globals() and not sensitivity_results.empty:
        sensitivity_pdf = sensitivity_results.copy().round(6)
        sensitivity_pdf = sensitivity_pdf.rename(columns={
            "dataset": "Dataset",
            "parameter": "Parametro",
            "n_RAB5": "N RAB5",
            "n_EV": "N EV",
            "p_value": "p-value",
            "r_biseriale": "r biseriale",
        })
        add_table_page(pdf, sensitivity_pdf, "Sensibilità Mann–Whitney U: con e senza 100626")

    # Pagina finale: test non parametrico RAB5 vs EV.
    if "mw_results" in globals() and not mw_results.empty:
        mw_pdf = mw_results.copy()
        mw_pdf["level"] = mw_pdf["level"].map({"cellula": "cellula (principale)", "curve": "curva (esplorativo)"}).fillna(mw_pdf["level"])
        mw_pdf = mw_pdf.rename(columns={
            "parameter": "Parametro",
            "level": "Livello",
            "n_RAB5": "N RAB5",
            "n_EV": "N EV",
            "median_RAB5": "Mediana RAB5",
            "median_EV": "Mediana EV",
            "U": "U",
            "p_value": "p-value",
            "rank_biserial_r": "r biseriale",
        })
        mw_pdf = mw_pdf[["Parametro", "Livello", "N RAB5", "N EV", "Mediana RAB5", "Mediana EV", "U", "p-value", "r biseriale"]]
        mw_pdf = mw_pdf.round({"Mediana RAB5": 4, "Mediana EV": 4, "U": 3, "p-value": 6, "r biseriale": 4})

        fig, ax = plt.subplots(figsize=(15, 6.5))
        ax.axis("off")
        ax.set_title("Test non parametrico: RAB5 vs EV", fontsize=14, pad=18)
        ax.text(0.5, 0.93, "Mann–Whitney U, bilaterale; livello cellula = analisi principale, livello curva = analisi esplorativa", ha="center", va="center", fontsize=10)
        table = ax.table(
            cellText=[[str(v) for v in row] for row in mw_pdf.itertuples(index=False)],
            colLabels=list(mw_pdf.columns), loc="center", cellLoc="center", bbox=[0.02, 0.18, 0.96, 0.62],
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.0, 1.5)
        for (row_index, _), cell in table.get_celld().items():
            if row_index == 0:
                cell.set_facecolor("#DDEEFF")
                cell.set_text_props(weight="bold")
            elif row_index > 0 and "cellula" in str(mw_pdf.iloc[row_index - 1]["Livello"]):
                cell.set_facecolor("#FFF3CD")
        ax.text(0.02, 0.08, "Le conclusioni inferenziali devono basarsi sulle righe a livello cellula per evitare pseudoreplicazione.", ha="left", va="center", fontsize=9)
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"PDF salvato: {pdf_path}")

PDF salvato: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_report_RAB5_EV.pdf


In [274]:
import os
import pandas as pd

# Usa il CSV del run corrente (quello creato dalla sezione 4 del notebook)
if "path_all" in globals() and isinstance(path_all, str) and os.path.exists(path_all):
    source_csv = path_all
else:
    source_csv = os.path.join(OUTPUT_DIR, "ting_summary_tutte_le_curve_RAB5_EV.csv")

if not os.path.exists(source_csv):
    raise FileNotFoundError(f"CSV sorgente non trovato: {source_csv}")

out_dir = os.path.dirname(source_csv)
out_csv = os.path.join(out_dir, "boxplot_finale_RAB5_EV_E0_betaE.csv")

df_box = pd.read_csv(source_csv)
df_box = df_box[df_box["macro_group"].isin(["RAB5", "EV"])].copy()

for col in ["E0_Pa", "betaE"]:
    if col in df_box.columns:
        df_box[col] = pd.to_numeric(df_box[col], errors="coerce")

cols_present = [c for c in ["E0_Pa", "betaE"] if c in df_box.columns]
if not cols_present:
    raise ValueError("Nessuna delle colonne E0_Pa / betaE trovata nel CSV sorgente.")

df_box = df_box[df_box[cols_present].notna().any(axis=1)].copy()

base_cols = ["macro_group", "group"]
for opt in ["cell_folder", "curve"]:
    if opt in df_box.columns:
        base_cols.append(opt)
export_cols = base_cols + cols_present

df_export = df_box[export_cols].sort_values(["macro_group", "group"] + [c for c in ["cell_folder", "curve"] if c in df_box.columns])
df_export.to_csv(out_csv, index=False)

print(f"Sorgente: {source_csv}")
print(f"CSV creato: {out_csv}")
print(f"Righe esportate: {len(df_export)}")
print(df_export[["macro_group"] + cols_present].groupby("macro_group").agg(["count", "median", "mean"]))

Sorgente: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/ting_summary_tutte_le_curve_RAB5_EV.csv
CSV creato: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/boxplot_finale_RAB5_EV_E0_betaE.csv
Righe esportate: 296
            E0_Pa                         betaE                    
            count      median        mean count    median      mean
macro_group                                                        
EV            129   82.959736  105.238575   129  0.170880  0.183657
RAB5          167  106.896797  127.334473   167  0.193015  0.204070


In [275]:
# Conteggio punti visualizzati nei boxplot per tipologia (RAB5/EV) e giornata
for macro_name in ["RAB5", "EV"]:
    sub = df[df["macro_group"] == macro_name].copy() if "macro_group" in df.columns else pd.DataFrame()
    if len(sub) == 0:
        continue
    print(f"\n=== {macro_name} ===")
    for col in PLOT_PARAMS.keys():
        total_points = int(sub[col].notna().sum()) if col in sub.columns else 0
        print(f"{col}: totale punti = {total_points}")
        by_group = sub.groupby("group")[col].apply(lambda s: int(s.notna().sum())).sort_index()
        print(by_group.to_string())


=== RAB5 ===
E0_Pa: totale punti = 167
group
090626_RAB5     3
100626_RAB5    36
150626_RAB5    35
160626_RAB5    30
250626_RAB5    38
260626_RAB5    25
betaE: totale punti = 167
group
090626_RAB5     3
100626_RAB5    36
150626_RAB5    35
160626_RAB5    30
250626_RAB5    38
260626_RAB5    25

=== EV ===
E0_Pa: totale punti = 129
group
090626_EV    40
100626_EV     4
130526_EV    19
140526_EV     5
250626_EV    26
260626_EV    35
betaE: totale punti = 129
group
090626_EV    40
100626_EV     4
130526_EV    19
140526_EV     5
250626_EV    26
260626_EV    35
